# UrbanNest – Data Foundation and Baseline Regression
**Goal:** clean the Ames Housing dataset, export a modeling-ready CSV, train a baseline Linear Regression model, and evaluate it with MAE and RMSE.

Dataset: [Ames Housing Dataset on Kaggle](https://www.kaggle.com/datasets/preathm/ames-housing-dataset)  
Target column: `SalePrice`  
This notebook is designed to run in **Google Colab**.

## Step 1 – Upload the dataset
Run this cell and select your `AmesHousing.csv` file from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose AmesHousing.csv

# Use the uploaded CSV file
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

filename = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if filename is None:
    raise ValueError("Please upload a CSV file, preferably AmesHousing.csv.")

df = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Loaded file:", filename)
print("Dataset shape:", df.shape)
df.head()

## Step 2 – Inspect the dataset
Check the number of rows and columns, data types, duplicates, and target column.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\\nData types:")
print(df.dtypes.value_counts())
print("\\nDuplicate rows:", df.duplicated().sum())
print("\\nTarget column exists:", "SalePrice" in df.columns)
print("\\nTarget missing values:", df["SalePrice"].isna().sum())
display(df.describe(include="all").T.head(20))

## Step 3 – Find missing values
This table shows only columns that contain missing values, sorted from most to fewest.

In [ ]:
missing_count = df.isna().sum()
missing_table = missing_count[missing_count > 0].sort_values(ascending=False).to_frame("Missing Values")
missing_table["Missing Percentage"] = (missing_table["Missing Values"] / len(df) * 100).round(2)
print("Columns containing missing values:", len(missing_table))
print("Total missing cells:", int(df.isna().sum().sum()))
display(missing_table)

## Step 4 – Visualize the target variable

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["SalePrice"].dropna(), bins=40, edgecolor="black")
plt.title("Distribution of House Sale Prices")
plt.xlabel("Sale Price")
plt.ylabel("Number of Houses")
plt.tight_layout()
plt.show()

## Step 5 – Prepare and clean the data
Cleaning rules:
- Remove rows only if the target `SalePrice` is missing. In this uploaded dataset it has no missing values.
- Drop `Order` and `PID` from model inputs because they are identifiers, not property characteristics.
- For categorical fields where missing means the feature is absent (such as no pool, garage, basement, alley, or fireplace), use the label `None`.
- Fill other categorical missing values with the most frequent category.
- For numeric fields where missing means the feature is absent (such as basement/garage area or veneer area), use `0`.
- Fill other numeric missing values with the median.
- One-hot encode categorical columns to create a numeric CSV.

**Important:** The exported cleaned CSV is for inspection/reuse. For model evaluation, we will fit learned preprocessing only on the training set to avoid data leakage.

In [ ]:
TARGET = "SalePrice"
if TARGET not in df.columns:
    raise ValueError("SalePrice target column is missing.")

# Work on a copy and remove rows only when the target is unavailable.
clean_df = df.dropna(subset=[TARGET]).copy()

# Identifiers are not useful predictors for a new property's price.
identifier_columns = [col for col in ["Order", "PID"] if col in clean_df.columns]
clean_df = clean_df.drop(columns=identifier_columns)

# Missing categorical value means the property does not have that feature.
no_feature_categorical = [
    "Alley", "Mas Vnr Type", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu", "Garage Type",
    "Garage Finish", "Garage Qual", "Garage Cond", "Pool QC", "Fence",
    "Misc Feature"
]

# Numeric columns where missing generally indicates that the feature is absent.
no_feature_numeric = [
    "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
    "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath",
    "Garage Yr Blt", "Garage Cars", "Garage Area"
]

categorical_columns = clean_df.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_columns = clean_df.select_dtypes(include=["number", "bool"]).columns.tolist()

# Handle categorical missing values.
for col in categorical_columns:
    if col in no_feature_categorical:
        clean_df[col] = clean_df[col].fillna("None")
    else:
        mode_values = clean_df[col].mode(dropna=True)
        fill_value = mode_values.iloc[0] if len(mode_values) else "Unknown"
        clean_df[col] = clean_df[col].fillna(fill_value)

# Handle numeric missing values.
for col in numeric_columns:
    if col in no_feature_numeric:
        clean_df[col] = clean_df[col].fillna(0)
    else:
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

print("Missing values after imputation:", int(clean_df.isna().sum().sum()))
print("Rows after cleaning:", clean_df.shape[0])
print("Columns before encoding:", clean_df.shape[1])
display(clean_df.head())

## Step 6 – Encode categorical variables and save cleaned CSV

In [ ]:
categorical_to_encode = clean_df.select_dtypes(include=["object", "category"]).columns.tolist()
cleaned_encoded_df = pd.get_dummies(
    clean_df,
    columns=categorical_to_encode,
    dtype=int
)

# Final quality checks
assert cleaned_encoded_df.isna().sum().sum() == 0, "There are still missing values."
assert "SalePrice" in cleaned_encoded_df.columns, "Target column was lost."

os.makedirs("data", exist_ok=True)
cleaned_csv_path = "data/cleaned_ames_housing.csv"
cleaned_encoded_df.to_csv(cleaned_csv_path, index=False)

print("Cleaned CSV saved to:", cleaned_csv_path)
print("Cleaned CSV shape:", cleaned_encoded_df.shape)
print("Missing values in cleaned CSV:", int(cleaned_encoded_df.isna().sum().sum()))
files.download(cleaned_csv_path)

## Step 7 – Prepare train and test data
We use 80% of the records for training and 20% for testing. `random_state=42` makes the split reproducible.

We keep the original categorical columns here. The Scikit-learn pipeline will learn imputers and one-hot encoding from the training data only.

In [ ]:
from sklearn.model_selection import train_test_split

model_df = df.dropna(subset=[TARGET]).copy()

# Remove ID columns from predictor variables.
X = model_df.drop(columns=[TARGET] + [c for c in ["Order", "PID"] if c in model_df.columns])
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Number of input features before encoding:", X_train.shape[1])

## Step 8 – Build preprocessing and Linear Regression model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

# Identify feature types from the training data.
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()

# Apply domain-based missing-value rules before the pipeline.
# These are fixed semantic replacements, not statistics learned from the test set.
def apply_structural_missing_rules(data):
    data = data.copy()
    for col in no_feature_categorical:
        if col in data.columns:
            data[col] = data[col].fillna("None")
    for col in no_feature_numeric:
        if col in data.columns:
            data[col] = data[col].fillna(0)
    return data

X_train_prepared = apply_structural_missing_rules(X_train)
X_test_prepared = apply_structural_missing_rules(X_test)

# Recalculate feature lists after the structural replacements.
categorical_features = X_train_prepared.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train_prepared.select_dtypes(include=["number", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

baseline_model.fit(X_train_prepared, y_train)
print("Linear Regression model trained successfully.")

## Step 9 – Predict and evaluate the model
MAE is the average absolute prediction error, in the same currency units as `SalePrice`. RMSE penalizes larger errors more strongly.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = baseline_model.predict(X_test_prepared)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("BASELINE LINEAR REGRESSION RESULTS")
print("----------------------------------")
print(f"MAE : ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²  : {r2:.4f}")

## Step 10 – Actual vs predicted chart

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.55)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], "r--")
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.title("UrbanNest: Actual vs Predicted House Prices")
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150)
plt.show()

## Step 11 – Identify important features
Permutation importance measures how much the model's MAE gets worse when one input feature is shuffled. Correlated features can share or distort importance, so interpret the ranking as an indication, not causation.

In [ ]:
from sklearn.inspection import permutation_importance

importance_result = permutation_importance(
    baseline_model,
    X_test_prepared,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test_prepared.columns,
    "MAE_Increase_When_Shuffled": importance_result.importances_mean
}).sort_values("MAE_Increase_When_Shuffled", ascending=False)

display(importance_df.head(15))

top_features = importance_df.head(15).sort_values("MAE_Increase_When_Shuffled")
plt.figure(figsize=(9, 7))
plt.barh(top_features["Feature"], top_features["MAE_Increase_When_Shuffled"])
plt.xlabel("Increase in MAE after shuffling")
plt.title("Top 15 Features – Permutation Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

## Step 12 – Save results and download project outputs

In [ ]:
# Save metrics
results = {
    "dataset": "Ames Housing Dataset (Kaggle)",
    "original_rows": int(df.shape[0]),
    "original_columns": int(df.shape[1]),
    "columns_with_missing_values": int(df.isna().sum().gt(0).sum()),
    "total_missing_cells": int(df.isna().sum().sum()),
    "cleaned_rows": int(cleaned_encoded_df.shape[0]),
    "cleaned_encoded_columns": int(cleaned_encoded_df.shape[1]),
    "missing_values_after_cleaning": int(cleaned_encoded_df.isna().sum().sum()),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "model": "Linear Regression",
    "MAE": float(mae),
    "RMSE": float(rmse),
    "R2": float(r2)
}

os.makedirs("results", exist_ok=True)
with open("results/baseline_results.json", "w") as f:
    json.dump(results, f, indent=4)

importance_df.to_csv("results/feature_importance.csv", index=False)

with open("results/baseline_results.txt", "w") as f:
    f.write("URBANNEST - BASELINE LINEAR REGRESSION RESULTS\\n")
    f.write(json.dumps(results, indent=4))
    f.write("\\n\\nTop 15 features:\\n")
    f.write(importance_df.head(15).to_string(index=False))

print(json.dumps(results, indent=4))
print("\\nSaved files:")
print("- data/cleaned_ames_housing.csv")
print("- results/baseline_results.json")
print("- results/baseline_results.txt")
print("- results/feature_importance.csv")
print("- actual_vs_predicted.png")
print("- feature_importance.png")

# Download the key deliverables from Colab.
files.download("results/baseline_results.txt")
files.download("results/feature_importance.csv")
files.download("actual_vs_predicted.png")
files.download("feature_importance.png")

## Step 13 – Conclusion
This project establishes the first baseline for UrbanNest. The cleaned CSV has no missing values and categorical variables have been encoded. The Linear Regression model's MAE and RMSE provide starting metrics that future models can be compared against.

**Submission deliverables:**
1. `data/cleaned_ames_housing.csv`
2. This Jupyter notebook (`.ipynb`)
3. `baseline_model.py` (optional if the notebook is the primary implementation; create/export it if the supervisor explicitly requires a script)
4. `README.md` describing dataset source, cleaning approach and MAE/RMSE.
